In [ ]:
!pip install -q transformers datasets scikit-learn torch

In [ ]:
import pandas as pd
import random
from transformers import pipeline

# ── 1. Load the custom CSV dataset ──────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

CSV_PATH = "/content/drive/MyDrive/aa_dataset-tickets-multi-lang-5-2-50-version.csv"
df = pd.read_csv(CSV_PATH)


# Collect all unique labels from the tag columns (tag_1 … tag_8)
tag_cols = [c for c in df.columns if c.startswith("tag_")]
labels = sorted(
    pd.concat([df[c] for c in tag_cols])
    .dropna()
    .unique()
    .tolist()
)

print("Dataset shape:", df.shape)
print("Sample row:")
print(df[["subject", "body"] + tag_cols[:3]].iloc[0].to_dict())
print("\nAll labels:", labels[:10], "...")

# ── 2. Build train / test splits ─────────────────────────────────────────────
# Use rows that have at least one tag as labelled data
labelled = df[df["tag_1"].notna()].reset_index(drop=True)

# 80/20 split (deterministic)
split = int(len(labelled) * 0.8)
train_data = labelled.iloc[:split]
test_data  = labelled.iloc[split:].reset_index(drop=True)

print(f"\nTrain size: {len(train_data)}, Test size: {len(test_data)}")

# ── 3. Helper: extract the true tags for one row ─────────────────────────────
def get_true_tags(row):
    """Return a list of the non-null tags for a dataset row."""
    return [row[c] for c in tag_cols if pd.notna(row[c])]

# ── 4. Load the LLM ──────────────────────────────────────────────────────────
llm = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

# ── 5. Zero-shot classifier ───────────────────────────────────────────────────
def zero_shot(ticket):
    prompt = f"""
You are a support ticket classifier.

Classify the ticket into 3 most relevant categories.

Possible categories:
{', '.join(labels)}

Ticket:
{ticket}

Return only top 3 tags separated by commas.
"""
    output = llm(prompt, max_length=50)
    return output[0]["generated_text"]

# ── 6. Few-shot classifier ────────────────────────────────────────────────────
def few_shot(ticket):
    prompt = f"""
You are a support ticket classification system.

Examples:

Ticket: I cannot log into my account
Tags: Account, Security, Authentication

Ticket: My card payment failed
Tags: Billing, Product, Bug

Ticket: App is very slow and lagging
Tags: Bug, Network, Disruption

Now classify this ticket:

Ticket: {ticket}

Return top 3 tags only.
"""
    output = llm(prompt, max_length=50)
    return output[0]["generated_text"]

# ── 7. Tag cleaner ───────────────────────────────────────────────────────────
def clean_tags(text):
    return [t.strip() for t in text.split(",")][:3]

# ── 8. Quick single-ticket demo ──────────────────────────────────────────────
sample_row = test_data.iloc[0]
sample_ticket = str(sample_row["subject"]) + ". " + str(sample_row["body"])

print("\nTicket:", sample_ticket[:200])
print("True tags:", get_true_tags(sample_row))

print("\nZero-shot:")
print(clean_tags(zero_shot(sample_ticket)))

print("\nFew-shot:")
print(clean_tags(few_shot(sample_ticket)))

# ── 9. Evaluate on 20 random samples ─────────────────────────────────────────
sample_size = 20
samples = test_data.sample(n=sample_size, random_state=42)

zero_preds  = []
few_preds   = []
true_labels = []

for _, item in samples.iterrows():
    text = str(item["subject"]) + ". " + str(item["body"])
    true = get_true_tags(item)              # list of true tags

    z = clean_tags(zero_shot(text))
    f = clean_tags(few_shot(text))

    zero_preds.append(z)
    few_preds.append(f)
    true_labels.append(true)

    print("Ticket:", text[:120])
    print("True:", true)
    print("Zero-shot:", z)
    print("Few-shot:", f)
    print("-" * 50)

# ── 10. Accuracy (any predicted tag matches any true tag) ────────────────────
def match_score(preds, true_labels):
    """1 point if any predicted tag overlaps with any true tag."""
    score = 0
    for pred, true in zip(preds, true_labels):
        if any(t in pred for t in true):
            score += 1
    return score / len(true_labels)

zero_acc = match_score(zero_preds, true_labels)
few_acc  = match_score(few_preds,  true_labels)

print("Zero-shot Accuracy:", zero_acc)
print("Few-shot Accuracy:", few_acc)

# ── 11. Results table ─────────────────────────────────────────────────────────
results_df = pd.DataFrame({
    "Method":   ["Zero-shot", "Few-shot"],
    "Accuracy": [zero_acc, few_acc]
})

print(results_df)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset shape: (28587, 16)
Sample row:
{'subject': 'Wesentlicher Sicherheitsvorfall', 'body': 'Sehr geehrtes Support-Team,\\n\\nich möchte einen gravierenden Sicherheitsvorfall melden, der gegenwärtig mehrere Komponenten unserer Infrastruktur betrifft. Betroffene Geräte umfassen Projektoren, Bildschirme und Speicherlösungen auf Cloud-Plattformen. Der Grund für die Annahme ist, dass der Vorfall eine potenzielle Datenverletzung im Zusammenhang mit einer Cyberattacke darstellt, was ein erhebliches Risiko für sensible Informationen und den laufenden Geschäftsbetrieb unserer Organisation bedeutet.\\n\\nUnsere initialen Untersuchungen haben ungewöhnliche Aktivitäten und Abweichungen bei den Geräten ergeben. Trotz der Umsetzung unserer standardisierten Behebungs- und Eindämmungsmaßnahmen konnte die Bedrohung bislang nicht vollständig eliminiert.', 'tag_1': 'Security

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

Ticket: Hilfe bei der Optimierung von Investmentstrategien für Data Analytics-Produkte. Könnten Sie Ratschläge zur Optimierung v
True: ['Feedback', 'Product', 'Feature', 'Tech Support']
Zero-shot: ['You are a support ticket classifier.\n\nClassify the ticket into 3 most relevant categories.\n\nPossible categories:\nAI', 'API', 'API Integration']
Few-shot: ['You are a support ticket classification system.\n\nExamples:\n\nTicket: I cannot log into my account\nTags: Account', 'Security', 'Authentication\n\nTicket: My card payment failed\nTags: Billing']
--------------------------------------------------


[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Ticket: Enhancing Digital Content with Adobe Premiere Pro 2021. Can you provide detailed advice on optimizing Adobe Premiere Pro
True: ['Feature', 'Documentation', 'Feedback']
Zero-shot: ['You are a support ticket classifier.\n\nClassify the ticket into 3 most relevant categories.\n\nPossible categories:\nAI', 'API', 'API Integration']
Few-shot: ['You are a support ticket classification system.\n\nExamples:\n\nTicket: I cannot log into my account\nTags: Account', 'Security', 'Authentication\n\nTicket: My card payment failed\nTags: Billing']
--------------------------------------------------


[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Ticket: nan. I am eager to learn about digital marketing strategies. Could you provide more details on how to enhance brand grow
True: ['Feedback', 'Sales', 'Lead']
Zero-shot: ['You are a support ticket classifier.\n\nClassify the ticket into 3 most relevant categories.\n\nPossible categories:\nAI', 'API', 'API Integration']
Few-shot: ['You are a support ticket classification system.\n\nExamples:\n\nTicket: I cannot log into my account\nTags: Account', 'Security', 'Authentication\n\nTicket: My card payment failed\nTags: Billing']
--------------------------------------------------
Zero-shot Accuracy: 0.0
Few-shot Accuracy: 0.2
      Method  Accuracy
0  Zero-shot       0.0
1   Few-shot       0.2
